# Session 4: The Aggregation Pipeline

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-04_pipeline_basics.py`

In [2]:
# Session 4 · Pipeline basics: $match, $group, $sort, $project, $limit, $count
from store_db import get_db, show

db = get_db()

print("1) Products per category with average and max price")
show(db.products.aggregate([
    {"$group": {"_id": "$category",
                "products": {"$sum": 1},
                "avgPrice": {"$avg": "$price"},
                "maxPrice": {"$max": "$price"}}},
    {"$sort": {"products": -1, "_id": 1}},
]))

print("\n2) Orders per status, but only non-cancelled ones ($match first)")
show(db.orders.aggregate([
    {"$match": {"status": {"$ne": "Cancelled"}}},
    {"$group": {"_id": "$status", "orders": {"$sum": 1}}},
    {"$sort": {"orders": -1, "_id": 1}},
]))

print("\n3) $project with computed fields: stock value, in whole thousands")
show(db.products.aggregate([
    {"$project": {"_id": 0, "name": 1,
                  "stockValue": {"$multiply": ["$price", "$stock"]},
                  "inThousands": {"$floor": {"$divide": [{"$multiply": ["$price", "$stock"]}, 1000]}}}},
    {"$sort": {"stockValue": -1}},
    {"$limit": 3},
]))

print("\n4) $count: how many Karachi customers")
show(db.customers.aggregate([{"$match": {"city": "Karachi"}}, {"$count": "karachiCustomers"}]))

print("\n5) $push / $addToSet collect values per group")
show(db.customers.aggregate([
    {"$group": {"_id": "$city", "names": {"$push": "$name"}, "tiers": {"$addToSet": "$loyalty"}}},
    {"$sort": {"_id": 1}},
]))

1) Products per category with average and max price
{"products": 4, "avgPrice": 4425.0, "maxPrice": 7500, "_id": "Accessories"}
{"products": 2, "avgPrice": 12500.0, "maxPrice": 18500, "_id": "Audio"}
{"products": 2, "avgPrice": 247500.0, "maxPrice": 285000, "_id": "Computers"}
{"products": 1, "avgPrice": 52000.0, "maxPrice": 52000, "_id": "Displays"}
{"products": 1, "avgPrice": 16000.0, "maxPrice": 16000, "_id": "Storage"}
(5 documents)

2) Orders per status, but only non-cancelled ones ($match first)
{"orders": 6, "_id": "Delivered"}
{"orders": 2, "_id": "Shipped"}
{"orders": 1, "_id": "Pending"}
(3 documents)

3) $project with computed fields: stock value, in whole thousands
{"name": "Ultrabook", "stockValue": 1050000, "inThousands": 1050}
{"name": "Gaming Laptop", "stockValue": 855000, "inThousands": 855}
{"name": "27-inch Monitor", "stockValue": 312000, "inThousands": 312}
(3 documents)

4) $count: how many Karachi customers
{"karachiCustomers": 3}
(1 document)

5) $push / $addToSe

## Example: `session-04_unwind_lookup.py`

In [3]:
# Session 4 · $unwind and $lookup: revenue reports across collections
from store_db import get_db, show

db = get_db()
delivered = {"$match": {"status": "Delivered"}}

print("1) $unwind turns one order with 2 items into 2 documents")
show(db.orders.aggregate([{"$match": {"_id": 5001}}, {"$unwind": "$items"},
                          {"$project": {"items": 1}}]))

print("\n2) Revenue per category from delivered orders ($unwind + $lookup + $group)")
show(db.orders.aggregate([
    delivered,
    {"$unwind": "$items"},
    {"$lookup": {"from": "products", "localField": "items.productId",
                 "foreignField": "_id", "as": "product"}},
    {"$unwind": "$product"},
    {"$group": {"_id": "$product.category",
                "units": {"$sum": "$items.qty"},
                "revenue": {"$sum": {"$multiply": ["$items.qty", "$items.price"]}}}},
    {"$sort": {"revenue": -1}},
]))

print("\n3) Top 3 customers by spend, with their names")
show(db.orders.aggregate([
    delivered,
    {"$unwind": "$items"},
    {"$group": {"_id": "$customerId",
                "spent": {"$sum": {"$multiply": ["$items.qty", "$items.price"]}},
                "orders": {"$addToSet": "$_id"}}},
    {"$lookup": {"from": "customers", "localField": "_id", "foreignField": "_id", "as": "c"}},
    {"$project": {"_id": 0, "name": {"$arrayElemAt": ["$c.name", 0]},
                  "city": {"$arrayElemAt": ["$c.city", 0]},
                  "spent": 1, "orderCount": {"$size": "$orders"}}},
    {"$sort": {"spent": -1}},
    {"$limit": 3},
]))

print("\n4) Products that were never part of a DELIVERED order")
show(db.products.aggregate([
    {"$lookup": {"from": "orders", "localField": "_id", "foreignField": "items.productId", "as": "o"}},
    {"$project": {"_id": 0, "name": 1,
                  "delivered": {"$filter": {"input": "$o", "as": "x",
                                            "cond": {"$eq": ["$$x.status", "Delivered"]}}}}},
    {"$match": {"delivered": {"$size": 0}}},
    {"$project": {"name": 1}},
]))

1) $unwind turns one order with 2 items into 2 documents
{"_id": 5001, "items": {"productId": 101, "qty": 2, "price": 1500}}
{"_id": 5001, "items": {"productId": 107, "qty": 1, "price": 4500}}
(2 documents)

2) Revenue per category from delivered orders ($unwind + $lookup + $group)
{"units": 1, "revenue": 210000, "_id": "Computers"}
{"units": 2, "revenue": 104000, "_id": "Displays"}
{"units": 2, "revenue": 32000, "_id": "Storage"}
{"units": 7, "revenue": 25500, "_id": "Accessories"}
(4 documents)

3) Top 3 customers by spend, with their names
{"spent": 243500, "name": "Hira Siddiqui", "city": "Karachi", "orderCount": 2}
{"spent": 67000, "name": "Ayesha Khan", "city": "Karachi", "orderCount": 2}
{"spent": 52000, "name": "Bilal Ahmed", "city": "Lahore", "orderCount": 1}
(3 documents)

4) Products that were never part of a DELIVERED order
{"name": "USB-C Hub"}
{"name": "Noise-Cancelling Headphones"}
{"name": "Bluetooth Speaker"}
{"name": "Gaming Laptop"}
(4 documents)


## Example: `session-04_reports.py`

In [4]:
# Session 4 · Reporting stages: $addFields, $cond, dates, $bucket, $facet, $out
from store_db import get_db, show

db = get_db()

print("1) Monthly sales (all non-cancelled orders)")
show(db.orders.aggregate([
    {"$match": {"status": {"$ne": "Cancelled"}}},
    {"$addFields": {"lineTotals": {"$map": {"input": "$items", "as": "i",
                                            "in": {"$multiply": ["$$i.qty", "$$i.price"]}}},
                    "month": {"$dateToString": {"format": "%Y-%m", "date": "$orderDate"}}}},
    {"$addFields": {"total": {"$sum": "$lineTotals"}}},
    {"$group": {"_id": "$month", "orders": {"$sum": 1}, "sales": {"$sum": "$total"}}},
    {"$sort": {"_id": 1}},
]))

print("\n2) $cond labels each product as Budget / Mid / Premium")
show(db.products.aggregate([
    {"$project": {"_id": 0, "name": 1, "price": 1,
                  "band": {"$cond": [{"$lt": ["$price", 10000]}, "Budget",
                           {"$cond": [{"$lt": ["$price", 100000]}, "Mid", "Premium"]}]}}},
    {"$sort": {"price": 1}},
]))

print("\n3) $bucket: how many products fall in each price range")
show(db.products.aggregate([
    {"$bucket": {"groupBy": "$price", "boundaries": [0, 5000, 20000, 100000, 1000000],
                 "default": "other", "output": {"count": {"$sum": 1}, "names": {"$push": "$name"}}}},
]))

print("\n4) $facet: several summaries in one pass")
show(db.products.aggregate([
    {"$facet": {
        "outOfStock": [{"$match": {"stock": 0}}, {"$project": {"_id": 0, "name": 1}}],
        "byBrand":    [{"$group": {"_id": "$specs.brand", "n": {"$sum": 1}}}, {"$match": {"n": {"$gt": 1}}}],
        "priceStats": [{"$group": {"_id": None, "min": {"$min": "$price"}, "max": {"$max": "$price"},
                                   "avg": {"$avg": "$price"}}}],
    }},
]))

print("\n5) $out writes the result to a new collection (a materialised report)")
list(db.orders.aggregate([{"$match": {"status": "Delivered"}},
                          {"$project": {"customerId": 1, "orderDate": 1}},
                          {"$out": "delivered_orders"}]))
print("collections now:", sorted(db.list_collection_names()))
print("documents in delivered_orders:", db.delivered_orders.count_documents({}))

1) Monthly sales (all non-cancelled orders)
{"orders": 3, "sales": 84500, "_id": "2025-03"}
{"orders": 2, "sales": 41000, "_id": "2025-04"}
{"orders": 3, "sales": 237100, "_id": "2025-05"}
{"orders": 1, "sales": 59500, "_id": "2025-06"}
(4 documents)

2) $cond labels each product as Budget / Mid / Premium
{"name": "Wireless Mouse", "price": 1500, "band": "Budget"}
{"name": "USB-C Hub", "price": 4200, "band": "Budget"}
{"name": "Webcam HD", "price": 4500, "band": "Budget"}
{"name": "Bluetooth Speaker", "price": 6500, "band": "Budget"}
{"name": "Mechanical Keyboard", "price": 7500, "band": "Budget"}
{"name": "Portable SSD 1TB", "price": 16000, "band": "Mid"}
{"name": "Noise-Cancelling Headphones", "price": 18500, "band": "Mid"}
{"name": "27-inch Monitor", "price": 52000, "band": "Mid"}
{"name": "Ultrabook", "price": 210000, "band": "Premium"}
{"name": "Gaming Laptop", "price": 285000, "band": "Premium"}
(10 documents)

3) $bucket: how many products fall in each price range
{"count": 3, "